# MQTT dev testing (pub/sub)

Ad-hoc publish/subscribe against the broker + creds already in `device_config.json`, for poking a claimed device without shelling out to `mosquitto_pub`/`mosquitto_sub` each time. Host-side only — no MicroPython kernel needed, plain `python3`.

Uses `certs/ca.pem` + `certs/client.pem` + `certs/private.pem` (same bundle used for manual `mosquitto_pub` testing) for the mTLS handshake.

In [1]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent
config = json.loads((REPO_ROOT / "device_config.json").read_text())

BROKER = config["mqtt_broker"]
PORT = config["mqtt_port"]
USERNAME = config["mqtt_username"]
PASSWORD = config["mqtt_password"]
TOPIC_PUB = config["mqtt_topic_pub"]
TOPIC_SUB = config["mqtt_topic_sub"]
TOPIC_STATUS = config["mqtt_topic_status"]

CA_CERT = REPO_ROOT / "certs" / "ca.pem"
CLIENT_CERT = REPO_ROOT / "certs" / "client.pem"
CLIENT_KEY = REPO_ROOT / "certs" / "private.pem"

print(f"broker={BROKER}:{PORT} username={USERNAME}")
print(f"pub={TOPIC_PUB} sub={TOPIC_SUB} status={TOPIC_STATUS}")

broker=192.168.1.216:8883 username=dev_fa6648eb
pub=devices/dev_fa6648eb/sensors sub=devices/dev_fa6648eb/commands status=devices/dev_fa6648eb/status


In [2]:
import paho.mqtt.client as mqtt

received = []


def _on_connect(client, userdata, flags, rc):
    print("connected, rc =", rc)


def _on_message(client, userdata, msg):
    payload = msg.payload.decode(errors="replace")
    received.append((msg.topic, payload))
    print(f"[{msg.topic}] {payload}")


client = mqtt.Client(client_id=f"{USERNAME}-notebook")
client.username_pw_set(USERNAME, PASSWORD)
client.tls_set(
    ca_certs=str(CA_CERT),
    certfile=str(CLIENT_CERT),
    keyfile=str(CLIENT_KEY),
)
client.on_connect = _on_connect
client.on_message = _on_message

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()

connected, rc = 0


## Subscribe

Listen to everything the device publishes (status reports + its own sensor readings). Run once — stays live in the background thread; messages print as they arrive and land in `received`.

In [3]:
client.subscribe(f"{TOPIC_STATUS}/#")
client.subscribe(f"{TOPIC_PUB}/#")

(0, 2)

## Publish commands

Adjust `SUFFIX`/`COMMAND` and rerun. Accepted commands per `RuntimeService._decode_command`: `"on"` / `"off"` / `"toggle"` (case-insensitive), or JSON `{"state": "on"}`.

In [4]:
SUFFIX = "relay"
COMMAND = "on"

topic = f"{TOPIC_SUB}/{SUFFIX}"
client.publish(topic, COMMAND)
print("published", COMMAND, "->", topic)

published on -> devices/dev_fa6648eb/commands/relay
[devices/dev_fa6648eb/status/relay] {"device": "ESP32 Starter Kit", "ok": true, "state": "on", "timestamp": 840341578, "timestamp_local": "2026-08-18T04:12:58+00:00", "action": "state_report", "timezone": "UTC", "client_id": "a71956c3-8133-4db2-9709-1b410ce62392"}


## Generic pub/sub (arbitrary topic)

For anything not covered above — OTA, log-level override, health report, etc.

In [5]:
TOPIC = "device/microweaver/log-level"
PAYLOAD = "debug"

client.publish(TOPIC, PAYLOAD)
print("published", PAYLOAD, "->", TOPIC)

published debug -> device/microweaver/log-level
[devices/dev_fa6648eb/status/relay] {"device": "ESP32 Starter Kit", "ok": true, "state": "on", "timestamp": 840341772, "timestamp_local": "2026-08-18T04:16:12+00:00", "action": "state_report", "timezone": "UTC", "client_id": "a71956c3-8133-4db2-9709-1b410ce62392"}
[devices/dev_fa6648eb/status/relay] {"device": "ESP32 Starter Kit", "ok": true, "state": "on", "timestamp": 840341780, "timestamp_local": "2026-08-18T04:16:20+00:00", "action": "state_report", "timezone": "UTC", "client_id": "a71956c3-8133-4db2-9709-1b410ce62392"}
[devices/dev_fa6648eb/status/relay] {"device": "ESP32 Starter Kit", "ok": true, "state": "on", "timestamp": 840341782, "timestamp_local": "2026-08-18T04:16:22+00:00", "action": "state_report", "timezone": "UTC", "client_id": "a71956c3-8133-4db2-9709-1b410ce62392"}
[devices/dev_fa6648eb/status/relay] {"device": "ESP32 Starter Kit", "ok": true, "state": "on", "timestamp": 840341795, "timestamp_local": "2026-08-18T04:16:3

## Disconnect

In [ ]:
client.loop_stop()
client.disconnect()